# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [4]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
import os
import json
__import__("pysqlite3")
import sys
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
OPENAI_API_KEY="voc-37530322316886551795186a355f97854fa7.94470412"
TAVILY_API_KEY="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"
OPENAI_BASE_URL="https://openai.vocareum.com/v1"

In [5]:
load_dotenv()

True

### VectorDB Instance

In [6]:
chroma_client = chromadb.PersistentClient(path="./chromadb")

### Collection

In [7]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

In [11]:
chroma_client.delete_collection(name="udaplay")

collection = chroma_client.create_collection(
   name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [12]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [13]:
def search_games(collection, query, n_results=3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    return {
        "documents": results.get("documents", []),
        "metadatas": results.get("metadatas", []),
        "ids": results.get("ids", []),
        "distances": results.get("distances", [])
    }


In [14]:
results = search_games(collection, "Who developed FIFA 21?", n_results=3)
print(results)

{'documents': [["[Xbox Series X|S] Halo Infinite (2021) - The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting.", '[PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.', '[PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.']], 'metadatas': [[{'Name': 'Halo Infinite', 'Publisher': 'Xbox Game Studios', 'Genre': 'First-person shooter', 'Platform': 'Xbox Series X|S', 'Description': "The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting.", 'YearOfRelease': 2021}, {'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.', 'Platform': 'PlayStation 1', 'Name': 'Gran Turismo', 'Publisher': 'Sony Computer Entertainment', 'Genre'